# Lexicon Generation : Word based vs BPE aware

Objective: Create a time-aware, domain-specific lexicon to represent news documents by capturing the correlation between specific words/tokens and stock price movements.
- **Time Window**: For each day $d$, we collect news articles from a 7-days look-back period to capture the impact of emerging financial terms.
- **Return Calculation**: We calculate the daily stock price variation ($\Delta$) for the day of each article's publication using the closing prices.
- **Word Scoring**: We assign each word $j$ a score $f(j)$ by averaging the $\Delta$ values of all articles in which that term appears.
- **Frequency Filtering**: We remove terms appearing in more than 90% of documents (too common) or fewer than 5 documents (statistically insignificant).
- **Marginal Screening**: By sorting remaining words by their average scores, we identify those consistently followed by significant positive or negative market variations.
- **Final Selection**: We form the lexicon by selecting words below the 20th percentile and above the 80th percentile to focus on the most impactful terms.

### Libraries

In [1]:
import pandas as pd
import spacy
from datetime import timedelta
from tqdm import tqdm
import os
import sys
from transformers import AutoTokenizer

sys.path.append(os.path.abspath(os.path.join("..")))
from src.lexicon_generation import (
    preprocess_spacy,
    build_daily_lexicon,
    visualize_daily_lexicon,
    ######################## BPE versions ########################
    preprocess_bpe,
    build_daily_lexicon_bpe,
    visualize_daily_lexicon_bpe,
)

c:\Users\dutau\Desktop\NLP\NLP_Financial_Event_Clustering_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Data Loading and Cleaning

In [2]:
# Load datasets
news = pd.read_csv("../data/raw/raw_news_2026.csv")
tweets = pd.read_csv("../data/raw/raw_tweets_2026.csv")
prices = pd.read_csv(
    "../data/raw/raw_nasdaq_2026.csv",
    skiprows=3,
    names=["date", "close", "high", "low", "open", "vol", "returns"],
)

In [3]:
news["date"] = pd.to_datetime(news["date"]).dt.date
prices["date"] = pd.to_datetime(prices["date"]).dt.date
prices_map = prices.set_index("date")["returns"].to_dict()

## **Word based approach** using SpaCy

### Lexicon generation

In [4]:
DTM_OUTPUT_DIR_SPACY = "../data/processed/daily_dtm_spacy/"
LEXICON_OUTPUT_DIR_SPACY = "../data/processed/daily_lexicons_full_spacy/"
FILTERED_LEXICON_OUTPUT_DIR_SPACY = "../data/processed/daily_lexicons_filtered_spacy/"

In [5]:
# Configuration
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
nlp.analyze_pipes(pretty=True)


============================= Pipeline Overview =============================

#   Component         Assigns       Requires   Scores        Retokenizes
-   ---------------   -----------   --------   -----------   -----------
0   tok2vec           doc.tensor                             False      
                                                                        
1   tagger            token.tag                tag_acc       False      
                                               pos_acc                  
                                               tag_micro_p              
                                               tag_micro_r              
                                               tag_micro_f              
                                                                        
2   attribute_ruler                                          False      
                                                                        
3   lemmatizer        token.lemma           

{'summary': {'tok2vec': {'assigns': ['doc.tensor'],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'tagger': {'assigns': ['token.tag'],
   'requires': [],
   'scores': ['tag_acc',
    'pos_acc',
    'tag_micro_p',
    'tag_micro_r',
    'tag_micro_f'],
   'retokenizes': False},
  'attribute_ruler': {'assigns': [],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'lemmatizer': {'assigns': ['token.lemma'],
   'requires': [],
   'scores': ['lemma_acc'],
   'retokenizes': False}},
 'problems': {'tok2vec': [],
  'tagger': [],
  'attribute_ruler': [],
  'lemmatizer': []},
 'attrs': {'doc.tensor': {'assigns': ['tok2vec'], 'requires': []},
  'token.lemma': {'assigns': ['lemmatizer'], 'requires': []},
  'token.tag': {'assigns': ['tagger'], 'requires': []}}}

In [6]:
print("Step 1: Pre-processing text...")
news["clean"] = (news["headline"] + " " + news["body"]).apply(
    lambda x: preprocess_spacy(x, nlp)
)
news.to_csv("../data/processed/news_2026_spacy.csv", index=False)

Step 1: Pre-processing text...


In [7]:
# Daily Loop (Rolling Window)
results = []
start_d = pd.to_datetime("2026-01-08").date()
end_d = pd.to_datetime("2026-03-31").date()

print("Step 2: Generating Daily Lexicons (Rolling Window)...")
for current_date in tqdm(pd.date_range(start_d, end_d)):
    d = current_date.date()
    # Collection of articles in window [d-7, d-1]
    window_news = news[
        (news["date"] >= d - timedelta(days=7)) & (news["date"] < d)
    ].copy()
    # Build Lexicon
    lex_map = build_daily_lexicon(
        window_news,
        prices_map,
        d,
        DTM_OUTPUT_DIR_SPACY,
        LEXICON_OUTPUT_DIR_SPACY,
        FILTERED_LEXICON_OUTPUT_DIR_SPACY,
    )

Step 2: Generating Daily Lexicons (Rolling Window)...


100%|██████████| 83/83 [00:04<00:00, 19.54it/s]


#### Vizualization

In [8]:
# Visualisation pour attaque d'Israel et USA en Iran:
visualize_daily_lexicon("2026-03-01")

## **BPE aware approach**

### Lexicon generation with BPE

In [9]:
DTM_OUTPUT_DIR_BPE = "../data/processed/daily_dtm_bpe/"
LEXICON_OUTPUT_DIR_BPE = "../data/processed/daily_lexicons_full_bpe/"
FILTERED_LEXICON_OUTPUT_DIR_BPE = "../data/processed/daily_lexicons_filtered_bpe/"

In [10]:
# Configuration du Tokenizer BPE (DistilRoBERTa)
print("Loading BPE Tokenizer...")
tokenizer_bpe = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-distilroberta-v1"
)
print(tokenizer_bpe)

Loading BPE Tokenizer...


RobertaTokenizer(name_or_path='sentence-transformers/all-distilroberta-v1', vocab_size=50265, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50264: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=False, special=True),
})


In [11]:
print("Step 1: Pre-processing text...")
text_to_process = news["headline"].fillna("") + " " + news["body"].fillna("")
# Tokenisation BPE
news["clean"] = text_to_process.apply(lambda x: preprocess_bpe(x, tokenizer_bpe))
news.to_csv("../data/processed/news_2026_bpe.csv", index=False)

Token indices sequence length is longer than the specified maximum sequence length for this model (754 > 512). Running this sequence through the model will result in indexing errors


Step 1: Pre-processing text...


In [12]:
# Daily Loop (Génération des lexiques)
print("Step 2: Generating Daily Lexicons (Rolling Window)...")
start_d = pd.to_datetime("2026-01-08").date()
end_d = pd.to_datetime("2026-03-31").date()

for current_date in tqdm(pd.date_range(start_d, end_d)):
    d = current_date.date()
    window_news = news[
        (news["date"] >= d - timedelta(days=7)) & (news["date"] < d)
    ].copy()

    # Appel à votre fonction (qui ne change pas, elle marche très bien)
    build_daily_lexicon_bpe(
        window_news,
        prices_map,
        d,
        DTM_OUTPUT_DIR_BPE,
        LEXICON_OUTPUT_DIR_BPE,
        FILTERED_LEXICON_OUTPUT_DIR_BPE,
    )

Step 2: Generating Daily Lexicons (Rolling Window)...


100%|██████████| 83/83 [00:04<00:00, 17.90it/s]


In [13]:
# Visualisation pour attaque d'Israel et USA en Iran:
visualize_daily_lexicon_bpe("2026-03-01")

##  Comparing Tokenisation approaches : SpaCy (Words) vs Voyage-4 (BPE)

Although both methods aim to extract the most impactful financial terms (Marginal Screening), they produce fundamentally different results due to their underlying design. Here is a comparative summary based on the execution of our pipeline.

| Criterion | Word-based Approach (SpaCy) | BPE Approach (Voyage-4 Nano) |
| :--- | :--- | :--- |
| **Base Unit** | The lemma (entire grammatical root of the word). | The subword, based on statistical probabilities. |
| **Token Examples (From lexicon)** | `surge`, `analyst`, `tuesday`, `technology` | `ĠTuesday`, `Ġwho`, `ing`, `ers`, `ĠNV` |
| **Human Interpretability** | **Excellent.** The charts display clear financial terms that are understandable at a glance. | **Low.** Presence of grammatical fragments (`ing`, `ed`) and space markers (`Ġ`), which are difficult to read in isolation. |
| **Applied Preprocessing** | Lemmatization, lowercasing, strict removal of stopwords and words < 2 characters. | Regex cleaning (removal of punctuation and numbers), but **preservation of case and stopwords** (`Ġwho`, `Ġby`). |
| **Execution Speed (DTM Generation)** | **~19.7 iterations/second.** Slower because spaCy performs linguistic analysis (even without parser/ner). | **~29.4 iterations/second.** Faster. HuggingFace's BPE (written in Rust) and the absence of lemmatization speed up the loop. |
| **Unknown Words (OOV)** | Ignored or poorly handled if absent from spaCy's dictionary. | Split into known sub-fragments (e.g., a newly coined company name will be parsed into 2-3 existing pieces). |
| **Case Sensitivity** | None (everything is converted to lowercase). | High (distinguishes `Ġmeta` from `ĠMeta` or `ĠApple` from `Ġapple`). |
| **P20 / P80 Thresholds (Example Feb-Mar 2026)** | P80: ~0.00022 <br> P20: -0.00065 | P80: ~0.00020 <br> P20: -0.00072 |
| **Objective / Best Use Case** | Traditional ML models (TF-IDF, Logistic Regression, Naive Bayes), pure lexical sentiment analysis. | Strict data preparation for Foundation Models (Transformers, LLMs) such as FinBERT or Voyage-4. |

> **Synthesis:** > The **SpaCy** approach creates a lexicon that is semantically perfect for human reading. The **BPE** approach, although visually chaotic with its `Ġ` prefixes, is the only mathematically valid method if the ultimate goal is to generate embeddings using LLMs, as it strictly adheres to the original training vocabulary of those models.